<a href="https://colab.research.google.com/github/ehsankarami1358/LOKA_HYDRO/blob/main/Unit2_Turbine_opening_analyse_R2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
UNIT 2 MONTHLY OPENING, OPERATING FREQUENCY, AND HILL-CHART ANALYSIS
===================================================================

Purpose
-------
For each available month in 2025 and 2026:

1. Divide operation into:
      - 2 MW power bins
      - 0.5 m net-head bins

2. Calculate for each MW/net-head operating cell:
      - number of samples
      - estimated operating hours
      - percentage of monthly operating time
      - median actual guide-vane opening
      - expected opening from the hill chart
      - actual minus hill-chart opening
      - monthly status

3. Identify the operating cells where the unit works most frequently.

4. Compare:
      - actual opening versus hill-chart opening
      - 2026 opening versus matched 2025 opening where common cells exist

Important
---------
Flow is NOT used in this analysis.

A higher actual opening than the hill-chart reference at the same MW and net head
can indicate that more guide-vane opening is required to produce the same power.
This is an indirect hydraulic-performance indicator and must be verified against:
sensor calibration, governor settings, wicket-gate linkage, operating mode,
head measurement, and maintenance history.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import LinearNDInterpolator
from __future__ import annotations

from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
from scipy.interpolate import LinearNDInterpolator

# ============================================================
# 1) INPUT FILES AND OUTPUT FOLDER
# ============================================================

FILE_2025 = r"/content/u2_MW_OP_FL_L_10_2_2025_17_3_2025_NEW_R3.csv"
FILE_2026 = r"/content/unıt2_MW_flow_4_5_2026_to_4_7_2026_R2.csv"
HILL_FILE = r"/content/LOKA_full_digital_hillchart_DB_with_opening_pct.xlsx"

OUTPUT_DIR = r"/content/data/unit2_monthly_opening_frequency_hillchart"

In [ ]:
!pip install xlsxwriter


# ============================================================
# 2) COLUMN NAMES
# ============================================================

TIMESTAMP_COL = "Timestamp"
POWER_COL = "ACTIVE_POWER(MW)"
OPENING_COL = "OPPENING(%)"
HEADWATER_COL = "HEAD_L(m)"
TAILWATER_COL = "TAIL_L(m)"
SPEED_COL = "SPEED(RPM)"


In [ ]:



POWER_BIN_WIDTH_MW = 1.0
HEAD_BIN_WIDTH_M = 0.5
MIN_POWER_MW = 20.0
MIN_OPENING_PCT = 1.0
MAX_OPENING_PCT = 100.0
NOMINAL_SPEED_RPM: Optional[float] = None
SPEED_TOLERANCE_RPM = 0.5
MIN_CELL_POINTS = 5
HIGH_CONFIDENCE_POINTS = 20
FREQUENT_MONTHLY_TIME_PCT = 3.0
VERY_FREQUENT_MONTHLY_TIME_PCT = 7.0
NORMAL_LIMIT_PP = 0.5
WARNING_LIMIT_PP = 1.5
ALARM_LIMIT_PP = 3.0
TOP_CELLS_PER_MONTH = 15
ANNOTATE_HEATMAPS = False


def read_csv_robust(path: str) -> pd.DataFrame:
    attempts = [
        {"encoding": "utf-8-sig", "sep": None, "engine": "python"},
        {"encoding": "utf-16", "sep": "\t"},
        {"encoding": "latin1", "sep": None, "engine": "python"},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if len(df.columns) > 1:
                return df
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not read {path}") from last_error


def load_year(path: str, year: int) -> pd.DataFrame:
    df = read_csv_robust(path)
    required = [TIMESTAMP_COL, POWER_COL, OPENING_COL, HEADWATER_COL, TAILWATER_COL, SPEED_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce", dayfirst=False)
    for c in [POWER_COL, OPENING_COL, HEADWATER_COL, TAILWATER_COL, SPEED_COL]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Power_MW"] = df[POWER_COL]
    df["Opening_pct"] = df[OPENING_COL]
    df["Speed_rpm"] = df[SPEED_COL]
    df["NetHead_m"] = df[HEADWATER_COL] - df[TAILWATER_COL]
    df = df.dropna(subset=["Timestamp", "Power_MW", "Opening_pct", "Speed_rpm", "NetHead_m"])
    df = df[df["Timestamp"].dt.year == year].sort_values("Timestamp").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"No valid {year} rows")
    df["Year"] = year
    df["Month"] = df["Timestamp"].dt.to_period("M").astype(str)
    df["Calendar_month"] = df["Timestamp"].dt.month
    return df


def select_stable(df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    running = (
        (df["Power_MW"] >= MIN_POWER_MW)
        & df["Opening_pct"].between(MIN_OPENING_PCT, MAX_OPENING_PCT)
    )
    nominal = NOMINAL_SPEED_RPM
    if nominal is None:
        nominal = float(df.loc[running, "Speed_rpm"].median())
    stable = running & df["Speed_rpm"].between(nominal - SPEED_TOLERANCE_RPM, nominal + SPEED_TOLERANCE_RPM)
    out = df.loc[stable].copy()
    if out.empty:
        raise ValueError("No stable operating rows")
    return out, nominal


def add_sample_hours(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for month, group in df.groupby("Month", sort=True):
        group = group.sort_values("Timestamp").copy()
        times = group["Timestamp"].drop_duplicates().sort_values()
        diffs = times.diff().dt.total_seconds().dropna()
        diffs = diffs[(diffs > 0) & (diffs <= 3600)]
        seconds = float(diffs.median()) if not diffs.empty else 60.0
        dup = group.groupby("Timestamp")["Timestamp"].transform("size")
        group["Sample_hours"] = seconds / 3600.0 / dup.clip(lower=1)
        parts.append(group)
    return pd.concat(parts, ignore_index=True)


def add_bins(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Power_bin_center_MW"] = (
        np.floor(out["Power_MW"] / POWER_BIN_WIDTH_MW) * POWER_BIN_WIDTH_MW
        + POWER_BIN_WIDTH_MW / 2
    )
    out["Head_bin_center_m"] = (
        np.floor(out["NetHead_m"] / HEAD_BIN_WIDTH_M) * HEAD_BIN_WIDTH_M
        + HEAD_BIN_WIDTH_M / 2
    )
    return out


def summarize_monthly(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["Year", "Month", "Calendar_month", "Head_bin_center_m", "Power_bin_center_MW"]
    s = (
        df.groupby(keys, observed=True)
        .agg(
            sample_count=("Opening_pct", "size"),
            operating_hours=("Sample_hours", "sum"),
            median_opening_pct=("Opening_pct", "median"),
            opening_std_pct=("Opening_pct", "std"),
            median_power_MW=("Power_MW", "median"),
            median_head_m=("NetHead_m", "median"),
        )
        .reset_index()
    )
    s["monthly_operating_pct"] = 100 * s["operating_hours"] / s.groupby("Month")["operating_hours"].transform("sum")
    s["monthly_frequency_rank"] = s.groupby("Month")["operating_hours"].rank(method="dense", ascending=False).astype(int)
    s["frequency_status"] = np.select(
        [s["monthly_operating_pct"] >= VERY_FREQUENT_MONTHLY_TIME_PCT,
         s["monthly_operating_pct"] >= FREQUENT_MONTHLY_TIME_PCT],
        ["Very frequent", "Frequent"],
        default="Occasional",
    )
    s["confidence"] = np.select(
        [s["sample_count"] >= HIGH_CONFIDENCE_POINTS, s["sample_count"] >= MIN_CELL_POINTS],
        ["High", "Medium"],
        default="Low",
    )
    s["usable"] = s["sample_count"] >= MIN_CELL_POINTS
    return s.sort_values(["Month", "monthly_frequency_rank"]).reset_index(drop=True)


# ============================================================
# OPERATING-ZONE PRIORITY SETTINGS
# ============================================================
TOP_ZONES = 15
IMPORTANT_SHARE_PCT = 1.0
MIN_COMPARISON_HOURS = 0.25
STABLE_LIMIT_PP = 0.5

# ============================================================
# 3) BUILD MONTHLY CELL DATA
# ============================================================

def prepare_monthly_data():
    raw_2025 = load_year(FILE_2025, 2025)
    raw_2026 = load_year(FILE_2026, 2026)

    stable_2025, nominal_2025 = select_stable(raw_2025)
    stable_2026, nominal_2026 = select_stable(raw_2026)

    stable_2025 = add_bins(add_sample_hours(stable_2025))
    stable_2026 = add_bins(add_sample_hours(stable_2026))

    combined = pd.concat([stable_2025, stable_2026], ignore_index=True)
    monthly = summarize_monthly(combined)

    return monthly, nominal_2025, nominal_2026


# ============================================================
# 4) MATCH TWO MONTHS AT THE SAME MW AND NET HEAD
# ============================================================

def compare_two_months(
    monthly: pd.DataFrame,
    previous_month: str,
    current_month: str,
) -> pd.DataFrame:
    keys = [
        "Head_bin_center_m",
        "Power_bin_center_MW",
    ]

    previous = monthly[
        (monthly["Month"] == previous_month)
        & monthly["usable"]
    ].copy()

    current = monthly[
        (monthly["Month"] == current_month)
        & monthly["usable"]
    ].copy()

    previous = previous[
        keys
        + [
            "median_opening_pct",
            "operating_hours",
            "monthly_operating_pct",
            "sample_count",
            "median_power_MW",
            "median_head_m",
        ]
    ].rename(
        columns={
            "median_opening_pct": "previous_opening_pct",
            "operating_hours": "previous_hours",
            "monthly_operating_pct": "previous_share_pct",
            "sample_count": "previous_points",
            "median_power_MW": "previous_actual_power_MW",
            "median_head_m": "previous_actual_head_m",
        }
    )

    current = current.merge(
        previous,
        on=keys,
        how="inner",
    )

    if current.empty:
        return current

    current["previous_month"] = previous_month
    current["current_month"] = current_month

    current["opening_change_pp"] = (
        current["median_opening_pct"]
        - current["previous_opening_pct"]
    )

    current["operating_share_change_pp"] = (
        current["monthly_operating_pct"]
        - current["previous_share_pct"]
    )

    current["matched_hours"] = np.minimum(
        current["operating_hours"],
        current["previous_hours"],
    )

    # Importance represents how much this operating cell matters in either month.
    current["importance_share_pct"] = np.maximum(
        current["monthly_operating_pct"],
        current["previous_share_pct"],
    )

    # Positive score = important zone with increasing opening.
    # Negative score = important zone with decreasing opening.
    current["signed_zone_priority"] = (
        current["importance_share_pct"]
        * current["opening_change_pp"]
    )

    # Only positive deterioration concern.
    current["positive_zone_concern"] = (
        current["importance_share_pct"]
        * current["opening_change_pp"].clip(lower=0)
    )

    current["zone_status"] = current[
        "opening_change_pp"
    ].map(classify_zone)

    current["work_shift_status"] = np.select(
        [
            current["operating_share_change_pp"] >= 1.0,
            current["operating_share_change_pp"] <= -1.0,
        ],
        [
            "Worked more in current month",
            "Worked less in current month",
        ],
        default="Similar operating share",
    )

    current["zone_conclusion"] = current.apply(
        build_zone_conclusion,
        axis=1,
    )

    current["important_zone"] = (
        (current["importance_share_pct"] >= IMPORTANT_SHARE_PCT)
        & (current["matched_hours"] >= MIN_COMPARISON_HOURS)
    )

    return current.sort_values(
        "positive_zone_concern",
        ascending=False,
    ).reset_index(drop=True)


def classify_zone(delta: float) -> str:
    if pd.isna(delta):
        return "No comparison"

    if abs(delta) <= STABLE_LIMIT_PP:
        return "Stable"

    if delta > ALARM_LIMIT_PP:
        return "Alarm - opening increased strongly"

    if delta > WARNING_LIMIT_PP:
        return "Warning - opening increased"

    if delta > STABLE_LIMIT_PP:
        return "Watch - opening increased slightly"

    if delta < -ALARM_LIMIT_PP:
        return "Opening decreased strongly"

    if delta < -WARNING_LIMIT_PP:
        return "Possible improvement"

    return "Opening decreased slightly"


def build_zone_conclusion(row: pd.Series) -> str:
    power = row["Power_bin_center_MW"]
    head = row["Head_bin_center_m"]
    opening_delta = row["opening_change_pp"]
    share_delta = row["operating_share_change_pp"]

    location = (
        f"{power:.1f} MW and {head:.2f} m net head"
    )

    if abs(opening_delta) <= STABLE_LIMIT_PP:
        opening_text = (
            f"opening remained stable ({opening_delta:+.2f} pp)"
        )
    elif opening_delta > 0:
        opening_text = (
            f"opening increased by {opening_delta:.2f} pp, "
            "indicating a potentially worse hydraulic trend"
        )
    else:
        opening_text = (
            f"opening decreased by {abs(opening_delta):.2f} pp, "
            "indicating a potentially improved hydraulic trend"
        )

    if share_delta >= 1.0:
        work_text = (
            f"and the unit worked {share_delta:.2f} percentage points "
            "more of the month in this zone"
        )
    elif share_delta <= -1.0:
        work_text = (
            f"and the unit worked {abs(share_delta):.2f} percentage points "
            "less of the month in this zone"
        )
    else:
        work_text = (
            "and the operating share remained approximately unchanged"
        )

    if opening_delta > STABLE_LIMIT_PP and share_delta > 0:
        final_text = (
            "This is a high-priority worse zone because both opening "
            "requirement and operating exposure increased."
        )
    elif opening_delta > STABLE_LIMIT_PP and share_delta <= 0:
        final_text = (
            "The zone became hydraulically worse, but exposure did not increase."
        )
    elif opening_delta < -STABLE_LIMIT_PP and share_delta > 0:
        final_text = (
            "The unit worked more in a zone showing a better opening trend."
        )
    elif opening_delta < -STABLE_LIMIT_PP:
        final_text = (
            "The zone shows improvement, although the unit used it less."
        )
    else:
        final_text = (
            "The zone is stable; operating redistribution is the main change."
        )

    return (
        f"At {location}, {opening_text}; {work_text}. {final_text}"
    )


# ============================================================
# 5) CREATE ALL MONTH-PAIR COMPARISONS
# ============================================================

def build_all_month_comparisons(
    monthly: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for year, year_data in monthly.groupby("Year"):
        months = sorted(year_data["Month"].unique())

        for previous_month, current_month in zip(
            months[:-1],
            months[1:],
        ):
            comparison = compare_two_months(
                monthly,
                previous_month,
                current_month,
            )

            if not comparison.empty:
                rows.append(comparison)

    # Also compare every 2026 month with aggregate 2025 cells.
    reference_2025 = (
        monthly[
            (monthly["Year"] == 2025)
            & monthly["usable"]
        ]
        .groupby(
            [
                "Head_bin_center_m",
                "Power_bin_center_MW",
            ],
            observed=True,
        )
        .agg(
            previous_opening_pct=("median_opening_pct", "median"),
            previous_hours=("operating_hours", "sum"),
            previous_share_pct=("monthly_operating_pct", "mean"),
            previous_points=("sample_count", "sum"),
            previous_actual_power_MW=("median_power_MW", "median"),
            previous_actual_head_m=("median_head_m", "median"),
        )
        .reset_index()
    )

    for month, current in monthly[
        (monthly["Year"] == 2026)
        & monthly["usable"]
    ].groupby("Month"):
        current = current.merge(
            reference_2025,
            on=[
                "Head_bin_center_m",
                "Power_bin_center_MW",
            ],
            how="inner",
        )

        if current.empty:
            continue

        current["previous_month"] = "2025 reference"
        current["current_month"] = month

        current["opening_change_pp"] = (
            current["median_opening_pct"]
            - current["previous_opening_pct"]
        )

        current["operating_share_change_pp"] = (
            current["monthly_operating_pct"]
            - current["previous_share_pct"]
        )

        current["matched_hours"] = np.minimum(
            current["operating_hours"],
            current["previous_hours"],
        )

        current["importance_share_pct"] = np.maximum(
            current["monthly_operating_pct"],
            current["previous_share_pct"],
        )

        current["signed_zone_priority"] = (
            current["importance_share_pct"]
            * current["opening_change_pp"]
        )

        current["positive_zone_concern"] = (
            current["importance_share_pct"]
            * current["opening_change_pp"].clip(lower=0)
        )

        current["zone_status"] = current[
            "opening_change_pp"
        ].map(classify_zone)

        current["work_shift_status"] = np.select(
            [
                current["operating_share_change_pp"] >= 1.0,
                current["operating_share_change_pp"] <= -1.0,
            ],
            [
                "Worked more in current month",
                "Worked less in current month",
            ],
            default="Similar operating share",
        )

        current["zone_conclusion"] = current.apply(
            build_zone_conclusion,
            axis=1,
        )

        current["important_zone"] = (
            (current["importance_share_pct"] >= IMPORTANT_SHARE_PCT)
            & (current["matched_hours"] >= MIN_COMPARISON_HOURS)
        )

        rows.append(current)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


# ============================================================
# 6) SUMMARY BY COMPARISON
# ============================================================

def build_comparison_summary(
    all_comparisons: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (
        previous_month,
        current_month,
    ), group in all_comparisons.groupby(
        ["previous_month", "current_month"]
    ):
        important = group[group["important_zone"]].copy()

        use = important if not important.empty else group

        weights = use["matched_hours"].clip(lower=0)

        if weights.sum() > 0:
            weighted_opening_change = float(
                np.average(
                    use["opening_change_pp"],
                    weights=weights,
                )
            )

            weighted_positive_concern = float(
                np.average(
                    use["opening_change_pp"].clip(lower=0),
                    weights=weights,
                )
            )
        else:
            weighted_opening_change = np.nan
            weighted_positive_concern = np.nan

        worse_exposure = float(
            use.loc[
                use["opening_change_pp"] > STABLE_LIMIT_PP,
                "monthly_operating_pct",
            ].sum()
        )

        stable_exposure = float(
            use.loc[
                use["opening_change_pp"].abs()
                <= STABLE_LIMIT_PP,
                "monthly_operating_pct",
            ].sum()
        )

        improved_exposure = float(
            use.loc[
                use["opening_change_pp"] < -STABLE_LIMIT_PP,
                "monthly_operating_pct",
            ].sum()
        )

        if weighted_positive_concern > ALARM_LIMIT_PP:
            overall_status = "Alarm"
        elif weighted_positive_concern > WARNING_LIMIT_PP:
            overall_status = "Warning"
        elif weighted_positive_concern > STABLE_LIMIT_PP:
            overall_status = "Watch"
        else:
            overall_status = "Stable / good"

        rows.append(
            {
                "Previous_period": previous_month,
                "Current_period": current_month,
                "matched_zones": len(group),
                "important_zones": len(important),
                "weighted_opening_change_pp":
                    weighted_opening_change,
                "weighted_positive_concern_pp":
                    weighted_positive_concern,
                "current_time_in_worse_zones_pct":
                    worse_exposure,
                "current_time_in_stable_zones_pct":
                    stable_exposure,
                "current_time_in_improved_zones_pct":
                    improved_exposure,
                "overall_status": overall_status,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# 7) CLEARER PLOTS
# ============================================================

def create_zone_priority_barplots(
    all_comparisons: pd.DataFrame,
    output_dir: Path,
) -> None:
    root = output_dir / "zone_priority_barplots"
    root.mkdir(parents=True, exist_ok=True)

    for (
        previous_month,
        current_month,
    ), group in all_comparisons.groupby(
        ["previous_month", "current_month"]
    ):
        important = group[group["important_zone"]].copy()

        if important.empty:
            important = group.copy()

        plot_data = important.nlargest(
            TOP_ZONES,
            "importance_share_pct",
        ).sort_values(
            "importance_share_pct",
            ascending=True,
        )

        labels = (
            plot_data["Power_bin_center_MW"]
            .map(lambda x: f"{x:.1f} MW")
            + " | "
            + plot_data["Head_bin_center_m"]
            .map(lambda x: f"{x:.2f} m")
        )

        plt.figure(figsize=(13, 8))
        bars = plt.barh(
            labels,
            plot_data["importance_share_pct"],
        )

        for bar, (_, row) in zip(
            bars,
            plot_data.iterrows(),
        ):
            annotation = (
                f" ΔOpening {row['opening_change_pp']:+.2f} pp"
                f" | ΔWork {row['operating_share_change_pp']:+.2f} pp"
                f" | {row['zone_status']}"
            )

            plt.text(
                bar.get_width() + 0.05,
                bar.get_y() + bar.get_height() / 2,
                annotation,
                va="center",
                fontsize=8,
            )

        plt.xlabel(
            "Importance: maximum monthly operating share (%)"
        )
        plt.ylabel("Matched 1 MW / 0.5 m operating zone")
        plt.title(
            f"{current_month} vs {previous_month}\n"
            "Frequently Used Zones with Opening and Work-Share Changes"
        )
        plt.tight_layout()

        safe_previous = previous_month.replace(" ", "_")
        plt.savefig(
            root
            / (
                f"{current_month}_vs_{safe_previous}_"
                "zone_priority.png"
            ),
            dpi=200,
        )
        plt.close()


def create_zone_status_exposure_plots(
    summary: pd.DataFrame,
    output_dir: Path,
) -> None:
    root = output_dir / "portfolio_plots"
    root.mkdir(parents=True, exist_ok=True)

    if summary.empty:
        return

    labels = (
        summary["Current_period"]
        + " vs "
        + summary["Previous_period"]
    )

    x = np.arange(len(summary))
    width = 0.25

    plt.figure(figsize=(13, 7))
    plt.bar(
        x - width,
        summary["current_time_in_improved_zones_pct"],
        width,
        label="Improved zones",
    )
    plt.bar(
        x,
        summary["current_time_in_stable_zones_pct"],
        width,
        label="Stable zones",
    )
    plt.bar(
        x + width,
        summary["current_time_in_worse_zones_pct"],
        width,
        label="Worse zones",
    )

    plt.xticks(
        x,
        labels,
        rotation=45,
        ha="right",
    )
    plt.ylabel("Current-month operating share (%)")
    plt.xlabel("Comparison")
    plt.title(
        "Where the Unit Worked: Improved, Stable, or Worse Matched Zones"
    )
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        root / "01_operating_share_by_zone_status.png",
        dpi=200,
    )
    plt.close()

    plt.figure(figsize=(13, 6))
    plt.plot(
        labels,
        summary["weighted_opening_change_pp"],
        marker="o",
        label="Weighted opening change",
    )
    plt.plot(
        labels,
        summary["weighted_positive_concern_pp"],
        marker="o",
        label="Positive opening concern",
    )
    plt.axhline(0, linewidth=1)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Opening change (percentage points)")
    plt.xlabel("Comparison")
    plt.title(
        "Matched-Zone Hydraulic Trend Weighted by Operating Exposure"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        root / "02_weighted_zone_trend.png",
        dpi=200,
    )
    plt.close()


def create_opening_and_share_comparison_bars(
    all_comparisons: pd.DataFrame,
    output_dir: Path,
) -> None:
    root = output_dir / "detailed_zone_comparisons"
    root.mkdir(parents=True, exist_ok=True)

    for (
        previous_month,
        current_month,
    ), group in all_comparisons.groupby(
        ["previous_month", "current_month"]
    ):
        important = group[group["important_zone"]].copy()

        if important.empty:
            continue

        plot_data = important.nlargest(
            10,
            "importance_share_pct",
        ).copy()

        plot_data["zone"] = (
            plot_data["Power_bin_center_MW"]
            .map(lambda x: f"{x:.1f} MW")
            + "\n"
            + plot_data["Head_bin_center_m"]
            .map(lambda x: f"{x:.2f} m")
        )

        x = np.arange(len(plot_data))
        width = 0.35

        plt.figure(figsize=(14, 7))
        plt.bar(
            x - width / 2,
            plot_data["previous_opening_pct"],
            width,
            label=previous_month,
        )
        plt.bar(
            x + width / 2,
            plot_data["median_opening_pct"],
            width,
            label=current_month,
        )

        for index, row in plot_data.reset_index(drop=True).iterrows():
            plt.text(
                index,
                max(
                    row["previous_opening_pct"],
                    row["median_opening_pct"],
                )
                + 0.3,
                f"Δ {row['opening_change_pp']:+.1f}",
                ha="center",
                fontsize=8,
            )

        plt.xticks(x, plot_data["zone"])
        plt.ylabel("Median opening (%)")
        plt.xlabel("Matched operating zone")
        plt.title(
            f"Opening Comparison at the Most Important Zones\n"
            f"{current_month} vs {previous_month}"
        )
        plt.legend()
        plt.tight_layout()

        safe_previous = previous_month.replace(" ", "_")
        plt.savefig(
            root
            / (
                f"{current_month}_vs_{safe_previous}_"
                "opening_bars.png"
            ),
            dpi=200,
        )
        plt.close()

        plt.figure(figsize=(14, 7))
        plt.bar(
            x - width / 2,
            plot_data["previous_share_pct"],
            width,
            label=previous_month,
        )
        plt.bar(
            x + width / 2,
            plot_data["monthly_operating_pct"],
            width,
            label=current_month,
        )

        plt.xticks(x, plot_data["zone"])
        plt.ylabel("Monthly operating share (%)")
        plt.xlabel("Matched operating zone")
        plt.title(
            f"Operating-Share Shift at the Most Important Zones\n"
            f"{current_month} vs {previous_month}"
        )
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            root
            / (
                f"{current_month}_vs_{safe_previous}_"
                "operating_share_bars.png"
            ),
            dpi=200,
        )
        plt.close()


# ============================================================
# 8) EXPORT
# ============================================================

def export_results(
    monthly: pd.DataFrame,
    all_comparisons: pd.DataFrame,
    summary: pd.DataFrame,
    output_dir: Path,
    nominal_2025: float,
    nominal_2026: float,
) -> Path:
    settings = pd.DataFrame(
        [
            ["Power-bin width", "1 MW"],
            ["Net-head-bin width", "0.5 m"],
            ["Important operating share (%)", IMPORTANT_SHARE_PCT],
            ["Minimum matched hours", MIN_COMPARISON_HOURS],
            ["Stable opening limit (pp)", STABLE_LIMIT_PP],
            ["Warning opening limit (pp)", WARNING_LIMIT_PP],
            ["Alarm opening limit (pp)", ALARM_LIMIT_PP],
            ["2025 nominal speed (rpm)", nominal_2025],
            ["2026 nominal speed (rpm)", nominal_2026],
            ["Flow meter", "Not used"],
            ["Hill chart", "Not used"],
            [
                "Better/worse definition",
                (
                    "Relative to historical opening required at the same "
                    "MW and net head; not direct efficiency."
                ),
            ],
        ],
        columns=["Parameter", "Value"],
    )

    report_path = (
        output_dir
        / "unit2_operating_zone_priority_report.xlsx"
    )

    with pd.ExcelWriter(
        report_path,
        engine="openpyxl",
    ) as writer:
        settings.to_excel(
            writer,
            index=False,
            sheet_name="Settings",
        )

        summary.to_excel(
            writer,
            index=False,
            sheet_name="Comparison_summary",
        )

        all_comparisons.to_excel(
            writer,
            index=False,
            sheet_name="Matched_zone_details",
        )

        all_comparisons[
            all_comparisons["important_zone"]
        ].to_excel(
            writer,
            index=False,
            sheet_name="Important_zones",
        )

        monthly.to_excel(
            writer,
            index=False,
            sheet_name="Monthly_cells",
        )

    summary.to_csv(
        output_dir / "comparison_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    all_comparisons.to_csv(
        output_dir / "matched_zone_details.csv",
        index=False,
        encoding="utf-8-sig",
    )

    all_comparisons[
        all_comparisons["important_zone"]
    ].to_csv(
        output_dir / "important_operating_zones.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return report_path


# ============================================================
# 9) MAIN
# ============================================================

def main() -> None:
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    monthly, nominal_2025, nominal_2026 = prepare_monthly_data()

    all_comparisons = build_all_month_comparisons(
        monthly
    )

    if all_comparisons.empty:
        raise ValueError(
            "No matched monthly operating zones were found."
        )

    summary = build_comparison_summary(
        all_comparisons
    )

    create_zone_priority_barplots(
        all_comparisons,
        output_dir,
    )

    create_zone_status_exposure_plots(
        summary,
        output_dir,
    )

    create_opening_and_share_comparison_bars(
        all_comparisons,
        output_dir,
    )

    report_path = export_results(
        monthly,
        all_comparisons,
        summary,
        output_dir,
        nominal_2025,
        nominal_2026,
    )

    print("=" * 80)
    print("UNIT 2 OPERATING-ZONE PRIORITY ANALYSIS")
    print("=" * 80)
    print(summary.to_string(index=False))
    print()
    print(f"Excel report: {report_path}")
    print(f"All plots and CSV files: {output_dir}")
    print()
    print(
        "Read the 'Important_zones' sheet first. A high-priority worse zone "
        "means the unit both worked more there and required more opening at "
        "the same MW and net head."
    )


if __name__ == "__main__":
    main()

UNIT 2 OPERATING-ZONE PRIORITY ANALYSIS
Previous_period Current_period  matched_zones  important_zones  weighted_opening_change_pp  weighted_positive_concern_pp  current_time_in_worse_zones_pct  current_time_in_stable_zones_pct  current_time_in_improved_zones_pct overall_status
 2025 reference        2026-05             52               11                   -0.284895                      0.238528                         0.329510                          2.329627                            1.882315  Stable / good
 2025 reference        2026-06             61               15                   -1.136390                      0.020195                         0.000000                          2.014540                           10.214010  Stable / good
 2025 reference        2026-07             46                8                   -2.471166                      0.162080                         3.294322                          0.000000                           15.528825  Stable / good
    